In [1]:
print("Howdy")

Howdy


In [13]:
import duckdb

conn = duckdb.connect()
conn.execute("INSTALL lance; LOAD lance;")
conn.execute("ATTACH './test_lancedb' AS lance_ns (TYPE LANCE, READ_WRITE);")

In [9]:


result = conn.execute("""
    SELECT feedback.cover_id, covers.cover_embedding
    FROM lance_ns.main.interactions AS feedback
    LEFT OUTER JOIN lance_ns.main.covers AS covers
      ON feedback.cover_id = covers.cover_id
""").pl()

# result = conn.execute("""
#     SELECT users.user_id, users.rowid as user_row_id, covers.cover_id, covers.cover_embedding, feedback.type, feedback.score
#     FROM lance_ns.main.interactions AS feedback
#     LEFT OUTER JOIN lance_ns.main.covers AS covers
#         ON (feedback.cover_id = covers.cover_id)    
#     LEFT OUTER JOIN lance_ns.main.users AS users
#         ON (feedback.user_id = users.user_id)
# """).pl()



# result = conn.execute("""
#     SELECT cover_id, _distance, _rowid, cover_embedding
#     FROM lance_vector_search(
#         'lance_ns.main.covers', 'cover_embedding',
#         [0.1, 0.2, 0.3]::FLOAT[3],
#         k = 5
#     )
#     ORDER BY _distance ASC;
# """).pl()

# result = conn.execute("""
#     SELECT cover_id FROM lance_ns.main.covers WHERE rowid IN (0, 999, 4);
# """).pl()



result

cover_id,cover_embedding
i64,"array[f32, 3]"
1,"[1.0, 3.0, 9.0]"
5,"[15.0, 2.0, 2.0]"
5,"[15.0, 2.0, 2.0]"
5,"[15.0, 2.0, 2.0]"


In [4]:
result

cover_id,cover_embedding
i64,"array[f32, 3]"
1,"[1.0, 3.0, 9.0]"
5,"[15.0, 2.0, 2.0]"
5,"[15.0, 2.0, 2.0]"
5,"[15.0, 2.0, 2.0]"


In [17]:
query = f"""
            SELECT users.rowid as user_row_id, covers.self.cid_field, covers.self.embedding_field, feedback.type, feedback.score
            FROM lance_ns.main.interactions AS feedback
            LEFT OUTER JOIN lance_ns.main.covers AS covers
                ON (feedback.self.cid_field = covers.self.cid_field)    
            LEFT OUTER JOIN lance_ns.main.users AS users
                ON (feedback.self.uid_field = users.self.uid_field)
            {f"WHERE feedback.timestamp >= timestamp '{21}'" if False else ""}
        """

print(query)


            SELECT users.rowid as user_row_id, covers.self.cid_field, covers.self.embedding_field, feedback.type, feedback.score
            FROM lance_ns.main.interactions AS feedback
            LEFT OUTER JOIN lance_ns.main.covers AS covers
                ON (feedback.self.cid_field = covers.self.cid_field)    
            LEFT OUTER JOIN lance_ns.main.users AS users
                ON (feedback.self.uid_field = users.self.uid_field)
            
        


In [15]:
query += "\nWHERE feedback.timestamp >= timestamp '{days_ago.strftime('%Y-%m-%d %H:%M:%S')}'"
print(query)


            SELECT users.rowid as user_row_id, covers.{self.cid_field}, covers.{self.embedding_field}, feedback.type, feedback.score
            FROM lance_ns.main.interactions AS feedback
            LEFT OUTER JOIN lance_ns.main.covers AS covers
                ON (feedback.{self.cid_field} = covers.{self.cid_field})    
            LEFT OUTER JOIN lance_ns.main.users AS users
                ON (feedback.{self.uid_field} = users.{self.uid_field})
        
WHERE feedback.timestamp >= timestamp '{days_ago.strftime('%Y-%m-%d %H:%M:%S')}'


In [7]:
import polars as pl

result.with_columns(
    pl.lit(8).alias("some_stuff")
)

cover_id,cover_embedding,some_stuff
i64,"array[f32, 3]",i32
1,"[1.0, 3.0, 9.0]",8
5,"[15.0, 2.0, 2.0]",8
5,"[15.0, 2.0, 2.0]",8
5,"[15.0, 2.0, 2.0]",8


In [36]:
import polars as pl
from enum import Enum

class FeedbackMap(tuple[int, int], Enum):
    rating = (0, 3)

feedback_dict = {e.name: e.value for e in FeedbackMap}

# def get_ratings_range(types: list[str]):
#     min_vals = []
#     max_vals = []
#     for t in types:
#         mapping = FeedbackMap[t].value
#         min_vals.append(mapping[0])
#         max_vals.append(mapping[1])
    
#     return pl.Series({"min_rating": min_vals, "max_rating": max_vals})

# def get_ratings_range(types: list[str]):
#     return pl.Series([FeedbackMap[t].value for t in types])

res = result.with_columns(
    pl.col("type").replace_strict(feedback_dict).list
        .to_struct(fields=["min_rating", "max_rating"])
        .alias("ratings_range"),
    #pl.col("cover_embedding").arr.to_list()
).unnest("ratings_range").drop(
    ["type", "user_id"]
).with_row_index()
res

index,user_row_id,cover_id,cover_embedding,score,min_rating,max_rating
u32,i64,i64,"array[f32, 3]",i64,i64,i64
0,0,5,"[15.0, 2.0, 2.0]",4,0,3
1,0,1,"[1.0, 3.0, 9.0]",4,0,3
2,1,1,"[1.0, 3.0, 9.0]",4,0,3


In [32]:
res["cover_embedding"].to_torch()

tensor([[15.,  2.,  2.],
        [ 1.,  3.,  9.],
        [ 1.,  3.,  9.]])

In [33]:
res.drop("cover_embedding").to_torch()

tensor([[0, 5, 4, 0, 3],
        [0, 1, 4, 0, 3],
        [1, 1, 4, 0, 3]])

In [100]:
feedback_dict

{'rating': (0, 3)}

In [14]:
print("yo")

yo


In [25]:
from enum import Enum

class FeedbackMap(tuple[int, int], Enum):
    Rating = (0, 3)

class HotRatingMap(int, Enum):
    Popular = FeedbackMap.Rating.value[1]
    Trending = FeedbackMap.Rating.value[1]

{k: v.value for k, v in HotRatingMap._member_map_.items()}

{'Popular': 3, 'Trending': 3}

In [26]:
{k: v.value for k, v in FeedbackMap._member_map_.items()}

{'Rating': (0, 3)}

In [22]:
_member_map_

NameError: name '_member_map_' is not defined

In [24]:
HotRatingMap.__members__

mappingproxy({'Popular': <HotRatingMap.Popular: 3>,
              'Trending': <HotRatingMap.Popular: 3>})

In [23]:
HotRatingMap._member_map_

{'Popular': <HotRatingMap.Popular: 3>, 'Trending': <HotRatingMap.Popular: 3>}

In [19]:
HotRatingMap._member_map_["Popular"].value

3

In [ ]:
def get_ratings_range(types: list[str]):
    return pl.Series([FeedbackMap[t].value for t in types])

result.with_columns(
    pl.col("type").map_batches(get_ratings_range, return_dtype=pl.List(pl.Int64)).list.to_struct(fields=["min_rating", "max_rating"])
    .alias("ratings_range")
).unnest("ratings_range")

In [43]:
result["user_row_id"].to_torch().unsqueeze(-1)

tensor([[0],
        [0],
        [1]])

In [31]:
result["cover_embedding"]

cover_embedding
"array[f32, 3]"
"[1.0, 3.0, 9.0]"
"[15.0, 2.0, 2.0]"


In [45]:
result["score"].to_torch()

tensor([4, 4, 4])

In [65]:
result.to_torch()

TypeError: cannot convert DataFrame to Tensor (mixed type columns result in `object` dtype)
Schema({'cover_id': Int64, 'cover_embedding': Array(Float32, shape=(3,))})

In [44]:
result.with_row_index()

index,user_id,cover_id,type,score,timestamp,cover_id_1,book_id,isbn_13,cover_url,cover_embedding,tower_embedding
u32,str,i64,str,i64,datetime[μs],i64,i64,str,str,"array[f32, 3]","array[f32, 2]"
0,"""b2c72cfc-7218-4362-a06a-6d4852…",1,"""rating""",4,2026-08-13 01:07:11.525132,1,2,"""1234567891011""","""something.cool.com/bruh.jpg""","[1.0, 3.0, 9.0]",null
1,"""2223305f-ae13-4cba-9c44-84b6be…",5,"""rating""",4,2026-08-13 01:02:46.654273,5,2,"""1234567891014""","""something.cool.com/bruh2.jpg""","[15.0, 2.0, 2.0]","[3.0, 2.0]"
2,"""2223305f-ae13-4cba-9c44-84b6be…",1,"""rating""",4,2026-08-13 01:06:54.856542,1,2,"""1234567891011""","""something.cool.com/bruh.jpg""","[1.0, 3.0, 9.0]",null


In [12]:
import polars as pl

result.with_row_index().filter(pl.col("index").is_in([2, 1]))

index,cover_id,cover_embedding,user_id
u32,i64,"array[f32, 3]",str
1,5,"[15.0, 2.0, 2.0]","""2223305f-ae13-4cba-9c44-84b6be…"
2,1,"[1.0, 3.0, 9.0]","""2223305f-ae13-4cba-9c44-84b6be…"


In [78]:
result["cover_id"].to_torch().unsqueeze(dim=-1)

tensor([[1],
        [5],
        [1]])

In [82]:
result["cover_embedding"].to_torch()

tensor([[ 1.,  3.,  9.],
        [15.,  2.,  2.],
        [ 1.,  3.,  9.]])

In [13]:
res = pl.DataFrame({"index": [2, 1]}).join(result.with_row_index(), on="index", how="left")
res

index,cover_id,cover_embedding,user_id
i64,i64,"array[f32, 3]",str
2,1,"[1.0, 3.0, 9.0]","""2223305f-ae13-4cba-9c44-84b6be…"
1,5,"[15.0, 2.0, 2.0]","""2223305f-ae13-4cba-9c44-84b6be…"


In [21]:
import uuid
import torch

def process_user_id(user_id: str) -> torch.Tensor:
    bytes_copy = bytearray(uuid.UUID(user_id).bytes_le)
    return (
        torch.frombuffer(bytes_copy, dtype=torch.int32)
        .to(dtype=torch.float32).unsqueeze(0)
    )

res["user_id"].map_elements(process_user_id).to_list()

[tensor([[ 5.7273e+08,  1.2873e+09, -1.2328e+09, -1.7732e+09]]),
 tensor([[ 5.7273e+08,  1.2873e+09, -1.2328e+09, -1.7732e+09]])]

user_id
str
"""2223305f-ae13-4cba-9c44-84b6be…"
"""2223305f-ae13-4cba-9c44-84b6be…"


In [69]:
res["cover_embedding"].to_torch()

tensor([[ 1.,  3.,  9.],
        [15.,  2.,  2.]])

In [71]:
res["cover_embedding"].to_torch().shape

torch.Size([2, 3])

In [61]:
import torch

torch.tensor([2]).repeat(3, 1)

tensor([[2],
        [2],
        [2]])